In [8]:
import cv2
import numpy as np
import tensorflow as tf

CLASS_COLORS = {
    1: [255,   0,   0],   # ship -> red
    2: [128, 128, 128],   # sargassum -> gray
    3: [255, 255, 255],   # oil -> white
}

# Load trained model
model = tf.keras.models.load_model("./segmentation_unet.h5", compile=False)

# Open video
cap = cv2.VideoCapture("../video/ManchaSat.mp4")

def preprocess_frame(frame):
    frame = cv2.resize(frame, (256, 256))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = gray[..., np.newaxis]
    return np.expand_dims(gray.astype("float32") / 255.0, axis=0)

def decode_mask(pred, frame_shape):
    mask = np.argmax(pred[0], axis=-1).astype(np.uint8)
    mask = cv2.resize(mask, (frame_shape[1], frame_shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask

def colorize_mask(mask):
    # Create a 3-channel RGB canvas
    mask_rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)

    # Ships = class 1 -> red
    mask_rgb[mask == 1] = (0, 0, 255)

    # Oil = class 6 -> white
    mask_rgb[mask == 3] = (255, 255, 255)

    # Sargassum = class 7 -> orange (BGR)
    mask_rgb[mask == 2] = (0, 165, 255)

    return mask_rgb

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    input_tensor = preprocess_frame(frame)
    pred = model.predict(input_tensor, verbose=0)

    mask = decode_mask(pred, frame.shape)
    mask_colored = colorize_mask(mask)

    oil_pixels = np.sum(mask == 3)

    cv2.putText(
        mask_colored,
        f"Area: {oil_pixels} px",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        1,
        cv2.LINE_AA,
    )

    # Show only the mask (not the original frame)
    cv2.imshow("Segmentation Mask", mask_colored)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
